# Notebook 10 — Model Contract & Fault-Onset Ground-Truth Audit

**Project:** PINN EV Battery / 100-cell software battery digital twin  
**Purpose:** reconcile actual training source, checkpoint shapes, normalizers, frozen Notebook 08 family split, and fault-onset provenance before any future inference or early-warning claim.

## Scientific guardrails

- Notebooks 04–08 are read-only historical artifacts. This notebook does not retrain models or edit the dataset.
- Notebook 09 is an audit, not proof that inference contracts or early-warning labels are valid.
- A detector that separates static faulty-cell labels is not automatically an early-warning model.
- No onset timestamp will be inferred from an anomaly score, fault-cell ID, scenario name, or first detector trigger.
- The 2.7 GB time-series CSV is inspected by schema, bounded samples, and optional chunked checks—not loaded wholesale.
- Missing or ambiguous evidence results in `NOT_EVALUABLE`, not a fabricated PASS.

This notebook can complete with blockers. Run from top to bottom in a fresh Colab runtime.

## 1. Runtime, Drive, and immutable run directory

The notebook records runtime versions, attempts to mount Drive, validates the project root, and writes to a unique timestamped run directory. If Drive is unavailable, it falls back to a local audit directory and records that project artifacts were unavailable.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, re, hashlib, platform, warnings, traceback, zipfile, ast as py_ast
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/content/drive/MyDrive/PINN_EV_Battery')
RUN_ID = datetime.now().strftime('run_%Y%m%d_%H%M%S')
DRIVE_OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'v1' / 'results_v1_model_contract_audit'

DRIVE_MOUNT_STATUS = 'NOT_ATTEMPTED'
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_MOUNT_STATUS = 'MOUNTED'
except Exception as e:
    DRIVE_MOUNT_STATUS = f'UNAVAILABLE: {type(e).__name__}: {e}'

if PROJECT_ROOT.is_dir():
    OUTPUT_ROOT = DRIVE_OUTPUT_ROOT
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
else:
    OUTPUT_ROOT = Path('/content') / 'PINN_EV_Battery_audit_local'
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'WARNING: project root not found: {PROJECT_ROOT}')
    print('Independent checks will continue; project artifacts will be NOT_EVALUABLE.')

RUN_DIR = OUTPUT_ROOT / RUN_ID
# Unique run path: do not overwrite a previous run if the clock string collides.
if RUN_DIR.exists():
    suffix = 1
    while (OUTPUT_ROOT / f'{RUN_ID}_{suffix}').exists(): suffix += 1
    RUN_DIR = OUTPUT_ROOT / f'{RUN_ID}_{suffix}'
RUN_DIR.mkdir(parents=True, exist_ok=False)

try:
    import torch
    TORCH_VERSION = torch.__version__
    CUDA_AVAILABLE = bool(torch.cuda.is_available())
    GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None
except Exception as e:
    torch = None
    TORCH_VERSION = f'UNAVAILABLE: {type(e).__name__}: {e}'
    CUDA_AVAILABLE = False
    GPU_NAME = None

RUN_CONFIG = {
    'notebook': '10_V1_ModelContract_and_FaultOnset_GroundTruth_FINAL',
    'run_id': RUN_DIR.name,
    'created_local': datetime.now().astimezone().isoformat(),
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'project_root': str(PROJECT_ROOT),
    'drive_mount_status': DRIVE_MOUNT_STATUS,
    'project_root_exists': PROJECT_ROOT.is_dir(),
    'run_directory': str(RUN_DIR),
    'python': sys.version,
    'platform': platform.platform(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'torch': str(TORCH_VERSION),
    'cuda_available': CUDA_AVAILABLE,
    'gpu_name': GPU_NAME,
    'policy': 'read-only audit; no model inference or training until contracts are verified'
}
(RUN_DIR / '10_run_config.json').write_text(json.dumps(RUN_CONFIG, indent=2, default=str), encoding='utf-8')
print(json.dumps(RUN_CONFIG, indent=2, default=str))

Mounted at /content/drive
{
  "notebook": "10_V1_ModelContract_and_FaultOnset_GroundTruth_FINAL",
  "run_id": "run_20261015_075459",
  "created_local": "2026-10-15T07:55:31.119574+00:00",
  "created_utc": "2026-10-15T07:55:31.119600+00:00",
  "project_root": "/content/drive/MyDrive/PINN_EV_Battery",
  "drive_mount_status": "MOUNTED",
  "project_root_exists": true,
  "run_directory": "/content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1_model_contract_audit/run_20261015_075459",
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "torch": "2.11.0+cu130",
  "cuda_available": true,
  "gpu_name": "Tesla T4",
  "policy": "read-only audit; no model inference or training until contracts are verified"
}


## 2. Artifact discovery and bounded inventory

Candidate paths are searched only inside known project subdirectories. Large CSVs are inventoried by size/schema without hashing their full contents. Small metadata, notebook, JSON, and checkpoint files are hashed where practical. Ambiguous candidates remain explicitly ambiguous.

In [2]:
EXPECTED_RELATIVE = {
    'cell_timeseries': ['data/v1/cell_timeseries.csv'],
    'scenario_instances': ['data/v1/scenario_instances.csv'],
    'b1_legacy': ['data/v1/results_v1/cell8_b1_best_model.pt'],
    'b3_legacy': ['data/v1/results_v1/B3_physics_informed_best.pt'],
    'b1_notebook06': ['data/v1/models_v1/B1_data_only_best.pt', 'data/v1/models_v1/B1_data_only_best.pt'],
    'b3_notebook06': ['data/v1/models_v1/B3_physics_informed_best.pt'],
    'legacy_normalizer': ['data/v1/results_v1/06_training_normalization_v2.json'],
    'notebook06_scaler': ['data/v1/models_v1/v1_scaler.json'],
    'notebook06_manifest': ['data/v1/results_v1/06_training_manifest.json'],
    'notebook05_target_spec': ['data/v1/05_future_target_specification.json'],
    'signal_scores': ['results_v1_localization/07_signal_localization_diagnostic.csv'],
    'notebook08_split_manifest': ['results_v1_localization/08_family_holdout_split_manifest.csv'],
    'notebook08_threshold_summary': ['results_v1_localization/08_family_holdout_threshold_summary.json'],
    'notebook08_metrics': ['results_v1_localization/08_family_holdout_threshold_metrics.csv'],
    'notebook08_cell_predictions': ['results_v1_localization/08_family_holdout_cell_predictions.csv'],
}
SEARCH_DIRS = [PROJECT_ROOT, PROJECT_ROOT / 'data' / 'v1', PROJECT_ROOT / 'data' / 'v1' / 'results_v1',
               PROJECT_ROOT / 'data' / 'v1' / 'models_v1', PROJECT_ROOT / 'results_v1_localization',
               PROJECT_ROOT / 'Colab Notebooks']

def locate_candidates(relpaths, filename=None):
    exact = [PROJECT_ROOT / p for p in relpaths if (PROJECT_ROOT / p).is_file()]
    exact = list(dict.fromkeys(exact))
    if exact: return exact
    if filename is None: return []
    found=[]
    for base in SEARCH_DIRS:
        if base.is_dir():
            try: found.extend([p for p in base.glob(filename) if p.is_file()])
            except Exception: pass
    return list(dict.fromkeys(found))

def sha256_file(path, chunk_size=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(chunk_size), b''): h.update(block)
    return h.hexdigest()

ARTIFACTS = {}
inventory=[]
for key, rels in EXPECTED_RELATIVE.items():
    filename=Path(rels[0]).name
    candidates=locate_candidates(rels, filename)
    ARTIFACTS[key]=candidates[0] if len(candidates)==1 else None
    if not candidates:
        inventory.append({'artifact_key':key,'status':'MISSING','candidate_count':0,'path':None,'size_bytes':None,'sha256':None})
    else:
        for p in candidates:
            try: size=p.stat().st_size
            except Exception: size=None
            digest=None; hash_status='NOT_HASHED'
            # Avoid expensive hashing of multi-gigabyte time-series files and large generated artifacts.
            if size is not None and size <= 250*1024*1024:
                try: digest=sha256_file(p); hash_status='HASHED'
                except Exception as e: hash_status=f'HASH_FAILED: {type(e).__name__}: {e}'
            inventory.append({'artifact_key':key,'status':'FOUND' if len(candidates)==1 else 'AMBIGUOUS',
                              'candidate_count':len(candidates),'path':str(p),'size_bytes':size,'sha256':digest,'hash_status':hash_status})

# Source notebooks are discovered by canonical name, preferring project Colab Notebooks and root locations.
NOTEBOOK_PATTERNS = {
 '04': ['04_V1_PyBaMM_DataGenerator_FINAL_CLEAN.ipynb','04_V1_PyBaMM_DataGenerator_FINAL.ipynb'],
 '05': ['05_V1_DatasetValidation_GroundTruth_FINAL_(3)(1).ipynb','05_V1_DatasetValidation_GroundTruth_FINAL.ipynb'],
 '06': ['06_V1_PINN_Training_ModelValidation_FINAL.ipynb'],
 '07': ['07_V1_AllCell_Localization_EarlyWarning_FINAL.ipynb'],
 '08': ['08_Family_HeldOut_Threshold_Calibration_Evaluation_FINAL.ipynb'],
 '09': ['09_V1_EarlyWarning_Prediction_FINAL.ipynb'],
}
NOTEBOOKS={}
for n, names in NOTEBOOK_PATTERNS.items():
    # The names are ordered by preference. Use the first filename with a unique match;
    # only fall back to the next historical alias when the preferred file is absent.
    selected=[]
    for name in names:
        hits=locate_candidates([], name)
        if hits:
            selected=list(dict.fromkeys(hits))
            break
    NOTEBOOKS[n]=selected
    inventory.append({'artifact_key':f'notebook_{n}_source','status':'FOUND' if len(NOTEBOOKS[n])==1 else ('AMBIGUOUS' if NOTEBOOKS[n] else 'MISSING'),
                      'candidate_count':len(NOTEBOOKS[n]),'path':str(NOTEBOOKS[n][0]) if len(NOTEBOOKS[n])==1 else None,
                      'size_bytes':NOTEBOOKS[n][0].stat().st_size if len(NOTEBOOKS[n])==1 else None,
                      'sha256':sha256_file(NOTEBOOKS[n][0]) if len(NOTEBOOKS[n])==1 and NOTEBOOKS[n][0].stat().st_size <= 250*1024*1024 else None})

inventory_df=pd.DataFrame(inventory)
inventory_df.to_csv(RUN_DIR/'10_artifact_inventory.csv',index=False)
print(inventory_df[['artifact_key','status','candidate_count','path','size_bytes']].to_string(index=False))

                artifact_key  status  candidate_count                                                                                                    path   size_bytes
             cell_timeseries   FOUND                1                                      /content/drive/MyDrive/PINN_EV_Battery/data/v1/cell_timeseries.csv 2694073736.0
          scenario_instances   FOUND                1                                   /content/drive/MyDrive/PINN_EV_Battery/data/v1/scenario_instances.csv       5802.0
                   b1_legacy   FOUND                1                        /content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1/cell8_b1_best_model.pt     140789.0
                   b3_legacy   FOUND                1                   /content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1/B3_physics_informed_best.pt     141499.0
               b1_notebook06   FOUND                1                          /content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1/B1_data_

## 3. Read-only source inspection: Notebook 04, 06, 08, and 09

This section parses notebook JSON and AST only; it does **not** execute source cells. It extracts the declared model contract and searches the generator for time-dependent fault-injection logic. Source notebooks not found in Drive are reported as unavailable.

In [3]:
import nbformat
SOURCE_REPORT = {}
SOURCE_TEXT = {}
for key, paths in NOTEBOOKS.items():
    if len(paths)!=1:
        SOURCE_REPORT[key]={'status':'NOT_EVALUABLE_SOURCE_MISSING_OR_AMBIGUOUS','candidate_paths':[str(p) for p in paths]}
        continue
    p=paths[0]
    try:
        nb=nbformat.read(str(p),as_version=4)
        code_cells=[(i,c.source) for i,c in enumerate(nb.cells) if c.cell_type=='code']
        md_cells=[(i,c.source) for i,c in enumerate(nb.cells) if c.cell_type=='markdown']
        joined='\n'.join(s for _,s in code_cells)
        SOURCE_TEXT[key]=joined
        SOURCE_REPORT[key]={'status':'READ_ONLY_SOURCE_PARSED','path':str(p),'cell_count':len(nb.cells),
                            'code_cell_count':len(code_cells),'markdown_cell_count':len(md_cells),
                            'code_sha256':hashlib.sha256(joined.encode()).hexdigest()}
    except Exception as e:
        SOURCE_REPORT[key]={'status':'SOURCE_PARSE_FAILED','path':str(p),'error':f'{type(e).__name__}: {e}'}

# Parse Notebook 06's declarative constants without executing notebook code.
NB06_CONTRACT={'status':'NOT_EVALUABLE_SOURCE_MISSING','input_features':None,'output_targets':None,'architecture':None,'horizons_seconds':None,'evidence':[]}
if '06' in SOURCE_TEXT:
    source=SOURCE_TEXT['06']
    try:
        nb=nbformat.read(str(NOTEBOOKS['06'][0]),as_version=4)
        cell_required={}; feature_keys=[]; target_keys=[]; input_expr=None
        for cell_i, cell in enumerate(nb.cells):
            if cell.cell_type!='code': continue
            try: tree=py_ast.parse(cell.source)
            except SyntaxError: continue
            for node in tree.body:
                if isinstance(node,(py_ast.Assign,py_ast.AnnAssign)):
                    targets=node.targets if isinstance(node,py_ast.Assign) else [node.target]
                    value=node.value
                    for target in targets:
                        if isinstance(target,py_ast.Name) and target.id=='CELL_REQUIRED':
                            try: cell_required=py_ast.literal_eval(value)
                            except Exception: pass
                        if isinstance(target,py_ast.Name) and target.id=='FEATURE_COLS' and isinstance(value,py_ast.List):
                            for el in value.elts:
                                if isinstance(el,py_ast.Subscript) and isinstance(el.value,py_ast.Name) and el.value.id=='CELL_REQUIRED':
                                    try: feature_keys.append(py_ast.literal_eval(el.slice))
                                    except Exception: pass
                        if isinstance(target,py_ast.Name) and target.id=='TARGET_COLS' and isinstance(value,py_ast.List):
                            for el in value.elts:
                                if isinstance(el,py_ast.Subscript) and isinstance(el.value,py_ast.Name) and el.value.id=='CELL_REQUIRED':
                                    try: target_keys.append(py_ast.literal_eval(el.slice))
                                    except Exception: pass
        input_features=[cell_required.get(k,k) for k in feature_keys] + (['horizon_s'] if 'INPUT_COLS = FEATURE_COLS + ["horizon_s"]' in source or "INPUT_COLS = FEATURE_COLS + ['horizon_s']" in source else [])
        output_targets=[f'{cell_required.get(k,k)}__future' for k in target_keys]
        arch='PredictiveMLP: input -> 128 -> 128 -> 128 -> output (Tanh after each of 3 hidden linear layers)' if 'class PredictiveMLP' in source and 'depth=3' in source else None
        target_spec_path=ARTIFACTS.get('notebook05_target_spec')
        horizons=None; horizons_source_status='NOT_VERIFIED_NO_TARGET_SPEC_JSON'
        if target_spec_path and Path(target_spec_path).is_file():
            try:
                target_spec_obj=json.loads(Path(target_spec_path).read_text(encoding='utf-8'))
                horizons_key=next((k for k in ['prediction_horizons_seconds','horizons_seconds','prediction_horizons'] if k in target_spec_obj),None)
                if horizons_key is not None:
                    horizons=[int(x) for x in target_spec_obj[horizons_key]]
                    horizons_source_status=f'READ_FROM_{horizons_key}_IN_{target_spec_path}'
            except Exception as e:
                horizons_source_status=f'TARGET_SPEC_PARSE_FAILED: {type(e).__name__}: {e}'
        NB06_CONTRACT={'status':'SOURCE_CONTRACT_EXTRACTED' if input_features and output_targets else 'PARTIAL_SOURCE_CONTRACT',
                       'input_features':input_features or None,'input_dimension':len(input_features) if input_features else None,
                       'output_targets':output_targets or None,'output_dimension':len(output_targets) if output_targets else None,
                       'architecture':arch,'horizons_seconds':horizons,'horizons_source_status':horizons_source_status,
                       'normalization_code':'X_mean/X_std and Y_mean/Y_std calculated from train_df; scaler saved as models_v1/v1_scaler.json',
                       'checkpoint_save_code':'B1_data_only_best.pt and B3_physics_informed_best.pt under models_v1 (inspect source cell for exact save expression)',
                       'evidence':[f'{NOTEBOOKS["06"][0]} code cells: CELL_REQUIRED, FEATURE_COLS, TARGET_COLS, INPUT_COLS, OUTPUT_COLS, PredictiveMLP, scaler save']}
    except Exception as e:
        NB06_CONTRACT={'status':'SOURCE_PARSE_FAILED','error':f'{type(e).__name__}: {e}'}

# Extract relevant generator snippets for evidence and future onset review.
GENERATOR_EVIDENCE=[]
if '04' in SOURCE_TEXT:
    s=SOURCE_TEXT['04']
    patterns=['TRAIN_FAULT_INSTANCES','VAL_FAULT_INSTANCES','TEST_FAULT_INSTANCES','is_fault = cell_id in fault_set','state_kind = "aged" if is_fault else "healthy"','run = degraded_run if is_fault else healthy_run','fault_cells_json']
    for pat in patterns:
        pos=s.find(pat)
        GENERATOR_EVIDENCE.append({'pattern':pat,'found':pos>=0,'character_offset':pos if pos>=0 else None,
                                   'excerpt':s[max(0,pos-220):pos+520] if pos>=0 else None})

source_summary={'source_notebooks':SOURCE_REPORT,'notebook06_contract':NB06_CONTRACT,'generator_evidence':GENERATOR_EVIDENCE}
(RUN_DIR/'10_source_code_audit.json').write_text(json.dumps(source_summary,indent=2,default=str),encoding='utf-8')
print('Notebook 06 source-derived contract:')
print(json.dumps(NB06_CONTRACT,indent=2,default=str))
print('\nGenerator fault chronology excerpts:')
for x in GENERATOR_EVIDENCE:
    print('\n',x['pattern'],'found=',x['found'],'\n',x['excerpt'])

Notebook 06 source-derived contract:
{
  "status": "NOT_EVALUABLE_SOURCE_MISSING",
  "input_features": null,
  "output_targets": null,
  "architecture": null,
  "horizons_seconds": null,
  "evidence": []
}

Generator fault chronology excerpts:


## 4. Checkpoint architecture and model-contract reconciliation

Checkpoints are inspected using `torch.load(..., weights_only=True)` when supported. The notebook never falls back to unsafe pickle loading. It extracts tensor shapes and metadata only. **No model inference is run here.** A dimension match is necessary but not sufficient: feature order, target meaning, scaler provenance, and preprocessing must also agree.

In [4]:
def safe_load_checkpoint(path):
    result={'path':str(path),'exists':Path(path).is_file(),'status':'NOT_EVALUABLE'}
    if not result['exists']: return result
    if torch is None:
        result.update(status='NOT_EVALUABLE_TORCH_UNAVAILABLE',note='Checkpoint not deserialized.')
        return result
    try:
        try:
            obj=torch.load(str(path),map_location='cpu',weights_only=True)
            load_mode='weights_only=True'
        except TypeError:
            result.update(status='NOT_EVALUABLE_SAFE_WEIGHTS_ONLY_UNSUPPORTED',
                          note='Installed PyTorch lacks weights_only support; refusing unsafe deserialization.')
            return result
        except Exception as e:
            result.update(status='SAFE_LOAD_FAILED',error=f'{type(e).__name__}: {e}',
                          note='No unsafe loading fallback attempted.')
            return result
        result['load_mode']=load_mode
        result['top_level_type']=type(obj).__name__
        state=obj if isinstance(obj,dict) and obj and all(hasattr(v,'shape') for v in obj.values()) else None
        if isinstance(obj,dict):
            result['top_level_keys']=[str(k) for k in obj.keys()][:100]
            for k in ['config','model_config','metadata','manifest','contract','input_columns','feature_names','output_columns','target_columns','horizons_seconds','params','scales']:
                if k in obj:
                    v=obj[k]
                    if isinstance(v,(str,int,float,bool,list,tuple,dict)):
                        try: json.dumps(v,default=str); result.setdefault('metadata_fields',{})[k]=v
                        except Exception: result.setdefault('metadata_fields',{})[k]=str(type(v).__name__)
            for k in ['state_dict','model_state_dict','model','weights']:
                if k in obj and isinstance(obj[k],dict) and any(hasattr(v,'shape') for v in obj[k].values()):
                    state=obj[k]; result['state_dict_container_key']=k; break
        if state is None and isinstance(obj,dict) and any(hasattr(v,'shape') for v in obj.values()): state=obj
        if state is None:
            result.update(status='SAFE_LOAD_NO_STATE_DICT',note='Loaded safely but no recognizable tensor state_dict was found.')
            return result
        tensors=[]
        for k,v in state.items():
            if hasattr(v,'shape'):
                shape=list(v.shape)
                tensors.append({'key':str(k),'shape':shape,'dtype':str(getattr(v,'dtype',None))})
        result['tensor_count']=len(tensors)
        result['tensor_shapes']=tensors[:300]
        linear=[x for x in tensors if len(x['shape'])==2 and x['key'].endswith('weight')]
        def layer_order(item):
            nums=re.findall(r'\d+',item['key'])
            return (int(nums[-1]) if nums else -1,item['key'])
        linear=sorted(linear,key=layer_order)
        result['linear_weight_layers']=linear
        if linear:
            result['input_dimension']=linear[0]['shape'][1]
            result['output_dimension']=linear[-1]['shape'][0]
            result['first_linear_weight_key']=linear[0]['key']
            result['last_linear_weight_key']=linear[-1]['key']
        result['status']='SAFE_STATE_DICT_INSPECTED'
        return result
    except Exception as e:
        result.update(status='CHECKPOINT_INSPECTION_ERROR',error=f'{type(e).__name__}: {e}')
        return result

# Discover checkpoints under canonical paths and search only relevant model/results folders for alternatives.
CHECKPOINT_CANDIDATES={
 'B1_legacy_reported': [PROJECT_ROOT/'data/v1/results_v1/cell8_b1_best_model.pt'],
 'B3_legacy_reported': [PROJECT_ROOT/'data/v1/results_v1/B3_physics_informed_best.pt'],
 'B1_notebook06': [PROJECT_ROOT/'data/v1/models_v1/B1_data_only_best.pt'],
 'B3_notebook06': [PROJECT_ROOT/'data/v1/models_v1/B3_physics_informed_best.pt'],
}
for k in list(CHECKPOINT_CANDIDATES):
    preferred=CHECKPOINT_CANDIDATES[k][0]
    if preferred.is_file():
        # Preserve the exact canonical path for each model version; same-named B3 files may exist in both folders.
        CHECKPOINT_CANDIDATES[k]=[preferred]
    else:
        name=preferred.name
        extras=[]
        for base in [PROJECT_ROOT/'data/v1', PROJECT_ROOT/'results_v1_localization']:
            if base.is_dir():
                try: extras.extend(base.rglob(name))
                except Exception: pass
        CHECKPOINT_CANDIDATES[k]=list(dict.fromkeys([p for p in extras if p.is_file()]))

CHECKPOINT_REPORTS={}
for label, paths in CHECKPOINT_CANDIDATES.items():
    if len(paths)==1:
        CHECKPOINT_REPORTS[label]=safe_load_checkpoint(paths[0])
    elif not paths:
        CHECKPOINT_REPORTS[label]={'path':None,'status':'MISSING','candidate_count':0}
    else:
        CHECKPOINT_REPORTS[label]={'path_candidates':[str(p) for p in paths],'status':'AMBIGUOUS_CHECKPOINT_PATH','candidate_count':len(paths)}

(RUN_DIR/'10_checkpoint_architecture_audit.json').write_text(json.dumps(CHECKPOINT_REPORTS,indent=2,default=str),encoding='utf-8')
print(json.dumps({k:{x:v.get(x) for x in ['path','status','input_dimension','output_dimension','first_linear_weight_key','last_linear_weight_key','top_level_keys','metadata_fields']} for k,v in CHECKPOINT_REPORTS.items()},indent=2,default=str))

{
  "B1_legacy_reported": {
    "path": "/content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1/cell8_b1_best_model.pt",
    "status": "SAFE_STATE_DICT_INSPECTED",
    "input_dimension": 7,
    "output_dimension": 1,
    "first_linear_weight_key": "network.0.weight",
    "last_linear_weight_key": "network.6.weight",
    "top_level_keys": [
      "model_state_dict",
      "input_dim",
      "output_dim",
      "hidden_width",
      "hidden_depth",
      "best_epoch",
      "best_val_loss"
    ],
    "metadata_fields": null
  },
  "B3_legacy_reported": {
    "path": "/content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1/B3_physics_informed_best.pt",
    "status": "SAFE_STATE_DICT_INSPECTED",
    "input_dimension": 7,
    "output_dimension": 1,
    "first_linear_weight_key": "network.0.weight",
    "last_linear_weight_key": "network.6.weight",
    "top_level_keys": [
      "model_state_dict",
      "input_columns",
      "output_columns",
      "seed",
      "architecture",
      "

## 5. Normalizer and target semantics audit

The legacy `06_training_normalization_v2.json` and Notebook 06 `models_v1/v1_scaler.json` are treated as distinct artifacts. This cell compares declared ordered columns with array lengths and checkpoint dimensions, records train-only provenance claims separately from verified evidence, and never truncates or reorders a scaler to force compatibility.

In [5]:
def read_json_safe(path):
    if not path or not Path(path).is_file(): return None, {'status':'MISSING','path':str(path) if path else None}
    try:
        obj=json.loads(Path(path).read_text(encoding='utf-8'))
        return obj, {'status':'READ','path':str(path),'top_level_type':type(obj).__name__,'top_level_keys':list(obj.keys()) if isinstance(obj,dict) else None}
    except Exception as e: return None, {'status':'JSON_PARSE_FAILED','path':str(path),'error':f'{type(e).__name__}: {e}'}

def first_present(obj, keys):
    if not isinstance(obj,dict): return None,None
    for k in keys:
        if k in obj: return k,obj[k]
    return None,None

def inspect_scaler(path):
    obj,base=read_json_safe(path)
    rep=dict(base)
    if not isinstance(obj,dict): return rep
    fields={
        'input_columns':['input_columns','input_cols','feature_names','input_features','X_columns','x_columns'],
        'output_columns':['output_columns','output_cols','target_columns','target_cols','output_targets','y_columns'],
        'x_mean':['X_mean','x_mean','input_mean','feature_mean','mean_x'],
        'x_scale':['X_std','x_std','input_std','feature_std','X_scale','x_scale'],
        'y_mean':['Y_mean','y_mean','target_mean','output_mean','mean_y'],
        'y_scale':['Y_std','y_std','target_std','output_std','Y_scale','y_scale'],
        'normalization_split':['normalization_split','fit_split','scaler_fit_split','split'],
        'target_name':['target_name','target','target_column','target_semantics'],
        'horizons':['horizons_seconds','prediction_horizons_seconds','supported_horizons','horizons'],
    }
    for dest, keys in fields.items():
        k,v=first_present(obj,keys)
        if k is not None: rep[dest]={'source_key':k,'value':v}
    for key in ['x_mean','x_scale','y_mean','y_scale']:
        item=rep.get(key)
        if item:
            try:
                arr=np.asarray(item['value'],dtype=float).reshape(-1)
                item['length']=int(arr.size); item['finite']=bool(np.isfinite(arr).all())
                if key in {'x_scale','y_scale'}: item['all_positive']=bool((arr>0).all())
                item['sample_values']=arr[:5].tolist()
            except Exception as e: item['length']=None; item['parse_error']=str(e)
    in_cols=rep.get('input_columns',{}).get('value') if isinstance(rep.get('input_columns'),dict) else None
    out_cols=rep.get('output_columns',{}).get('value') if isinstance(rep.get('output_columns'),dict) else None
    xmean=rep.get('x_mean',{}).get('length') if isinstance(rep.get('x_mean'),dict) else None
    xscale=rep.get('x_scale',{}).get('length') if isinstance(rep.get('x_scale'),dict) else None
    ymean=rep.get('y_mean',{}).get('length') if isinstance(rep.get('y_mean'),dict) else None
    yscale=rep.get('y_scale',{}).get('length') if isinstance(rep.get('y_scale'),dict) else None
    rep['declared_input_column_count']=len(in_cols) if isinstance(in_cols,list) else None
    rep['declared_output_column_count']=len(out_cols) if isinstance(out_cols,list) else None
    rep['x_array_lengths_consistent']=bool(xmean is not None and xscale is not None and xmean==xscale and (not isinstance(in_cols,list) or xmean==len(in_cols)))
    rep['y_array_lengths_consistent']=bool(ymean is not None and yscale is not None and ymean==yscale and (not isinstance(out_cols,list) or ymean==len(out_cols)))
    split=rep.get('normalization_split',{}).get('value') if isinstance(rep.get('normalization_split'),dict) else None
    rep['train_only_claim_found']=str(split).strip().lower() in {'train_only','train','training_only','train-only'}
    rep['train_only_provenance_verified']=False # a string claim is not provenance by itself
    rep['semantic_contract_status']='UNRESOLVED_UNTIL_MATCHED_TO_TRAINING_SOURCE_AND_CHECKPOINT'
    return rep

scaler_paths=[]
for p in [PROJECT_ROOT/'data/v1/results_v1/06_training_normalization_v2.json',PROJECT_ROOT/'data/v1/models_v1/v1_scaler.json']:
    if p.is_file(): scaler_paths.append(p)
# discover named normalizers in the versioned results area only
for base in [PROJECT_ROOT/'data/v1/results_v1',PROJECT_ROOT/'data/v1/models_v1']:
    if base.is_dir():
        try: scaler_paths.extend([p for p in base.glob('*normaliz*.json') if p.is_file()])
        except Exception: pass
scaler_paths=list(dict.fromkeys(scaler_paths))
SCALER_REPORTS={str(p):inspect_scaler(p) for p in scaler_paths}


manifest_obj, MANIFEST_REPORT=read_json_safe(ARTIFACTS.get('notebook06_manifest'))
if isinstance(manifest_obj,dict):
    MANIFEST_REPORT.update({k:manifest_obj.get(k) for k in ['notebook','status','input_columns','output_columns','prediction_horizons_seconds','data_policy','checks'] if k in manifest_obj})
    MANIFEST_REPORT['path']=str(ARTIFACTS.get('notebook06_manifest'))
source_text_06=SOURCE_TEXT.get('06','')
nb06_train_only_source=(
    'train_df = supervised[supervised["split"] == "train"].copy()' in source_text_06 and
    'X_mean = np.nanmean(X_train, axis=0)' in source_text_06 and
    'Y_mean = np.nanmean(Y_train, axis=0)' in source_text_06 and
    'v1_scaler.json' in source_text_06
)
source_features=NB06_CONTRACT.get('input_features')
source_targets=NB06_CONTRACT.get('output_targets')
manifest_inputs=manifest_obj.get('input_columns') if isinstance(manifest_obj,dict) else None
manifest_outputs=manifest_obj.get('output_columns') if isinstance(manifest_obj,dict) else None
manifest_horizons=manifest_obj.get('prediction_horizons_seconds') if isinstance(manifest_obj,dict) else None
manifest_contract_matches=(manifest_inputs==source_features and manifest_outputs==source_targets) if source_features and source_targets else False
manifest_horizons_match=(manifest_horizons==NB06_CONTRACT.get('horizons_seconds')) if manifest_horizons is not None and NB06_CONTRACT.get('horizons_seconds') is not None else False



# Link checkpoint dimensions to each scaler without modifying either artifact.
for label, cp in CHECKPOINT_REPORTS.items():
    cp['notebook06_source_contract']={'input_dimension':NB06_CONTRACT.get('input_dimension'),
                                      'output_dimension':NB06_CONTRACT.get('output_dimension'),
                                      'input_features':NB06_CONTRACT.get('input_features'),
                                      'output_targets':NB06_CONTRACT.get('output_targets')}
    cp['dimension_matches_notebook06_source']=(cp.get('input_dimension')==NB06_CONTRACT.get('input_dimension') and cp.get('output_dimension')==NB06_CONTRACT.get('output_dimension')) if cp.get('input_dimension') is not None else None
    cp['inference_authorized']=False
    cp['contract_status']='NOT_EVALUABLE_MODEL_CONTRACT'

# Build explicit per-model contract reports, retaining legacy and Notebook 06 candidates separately.
MODEL_CONTRACT_REPORTS={}
notebook06_scaler_rep=next((v for k,v in SCALER_REPORTS.items() if Path(k).name=='v1_scaler.json'),None)
scaler_inputs=notebook06_scaler_rep.get('input_columns',{}).get('value') if notebook06_scaler_rep and isinstance(notebook06_scaler_rep.get('input_columns'),dict) else None
scaler_outputs=notebook06_scaler_rep.get('output_columns',{}).get('value') if notebook06_scaler_rep and isinstance(notebook06_scaler_rep.get('output_columns'),dict) else None
scaler_arrays_ok=bool(notebook06_scaler_rep and notebook06_scaler_rep.get('x_array_lengths_consistent') and notebook06_scaler_rep.get('y_array_lengths_consistent'))
scaler_finite_ok=bool(notebook06_scaler_rep is not None and all(bool(notebook06_scaler_rep.get(k,{}).get('finite',False)) for k in ['x_mean','x_scale','y_mean','y_scale']))
scaler_scales_positive=bool(notebook06_scaler_rep and notebook06_scaler_rep.get('x_scale',{}).get('all_positive') is True and notebook06_scaler_rep.get('y_scale',{}).get('all_positive') is True)
scaler_order_matches=(scaler_inputs==source_features and scaler_outputs==source_targets)
for model_name in ['B1','B3']:
    candidates=[]
    for label in [f'{model_name}_legacy_reported',f'{model_name}_notebook06']:
        cp=CHECKPOINT_REPORTS.get(label,{})
        dim_match=cp.get('dimension_matches_notebook06_source') is True
        cp_path=cp.get('path') or ''
        canonical_path='/models_v1/' in cp_path.replace('\\','/')
        checks={
          'safe_state_dict_inspected':cp.get('status')=='SAFE_STATE_DICT_INSPECTED',
          'checkpoint_dimensions_match_source':dim_match,
          'canonical_notebook06_model_path':bool(canonical_path),
          'manifest_ordered_columns_match_source':bool(manifest_contract_matches),
          'scaler_ordered_columns_match_source':bool(scaler_order_matches),
          'scaler_array_lengths_consistent':bool(scaler_arrays_ok),
          'scaler_parameters_finite':bool(scaler_finite_ok),
          'scaler_scales_positive':bool(scaler_scales_positive),
          'train_only_scaling_code_present':bool(nb06_train_only_source),
          'manifest_horizons_match_target_spec':bool(manifest_horizons_match),
        }
        all_checks=all(checks.values())
        candidates.append({'candidate_id':label,'checkpoint_path':cp.get('path'),'checkpoint_status':cp.get('status'),
          'architecture_evidence':cp.get('linear_weight_layers'),'input_dimension':cp.get('input_dimension'),
          'ordered_input_features':source_features if dim_match and manifest_contract_matches else None,
          'output_dimension':cp.get('output_dimension'),'ordered_output_targets':source_targets if dim_match and manifest_contract_matches else None,
          'normalization_source':str(next((Path(k) for k in SCALER_REPORTS if Path(k).name=='v1_scaler.json'),'')) if scaler_order_matches else None,
          'normalization_parameters':notebook06_scaler_rep if scaler_order_matches else None,
          'target_name_and_meaning':'capacity_retention_soh__future is one of seven future-state outputs; model is multi-output, not SOH-only.' if source_targets and 'capacity_retention_soh__future' in source_targets else None,
          'supported_horizons_seconds':NB06_CONTRACT.get('horizons_seconds') if manifest_horizons_match else None,
          'training_validation_test_split_provenance':MANIFEST_REPORT.get('data_policy'),
          'contract_checks':checks,'checkpoint_manifest_cryptographic_binding':'NOT_FOUND_IN_AVAILABLE_METADATA',
          'contract_status':'CONTRACT_MATCHES_SOURCE_FOR_INFERENCE_SMOKE_TEST' if all_checks else 'NOT_EVALUABLE_MODEL_CONTRACT',
          'inference_executed':False})
    validated=[x for x in candidates if x['contract_status']=='CONTRACT_MATCHES_SOURCE_FOR_INFERENCE_SMOKE_TEST']
    MODEL_CONTRACT_REPORTS[model_name]={'model_identifier':model_name,'notebook06_source_contract':NB06_CONTRACT,
      'training_manifest':MANIFEST_REPORT,'candidate_checkpoints':candidates,
      'contract_status':'CONTRACT_MATCHES_SOURCE_FOR_INFERENCE_SMOKE_TEST' if validated else 'NOT_EVALUABLE_MODEL_CONTRACT',
      'inference_supported_for_smoke_test':bool(validated),
      'final_note':'This audit does not execute inference. A matched contract permits a separate bounded inference smoke test, not a claim of finite predictions or validated scientific performance.'}
    outname='10_b1_model_contract.json' if model_name=='B1' else '10_b3_model_contract.json'
    (RUN_DIR/outname).write_text(json.dumps(MODEL_CONTRACT_REPORTS[model_name],indent=2,default=str),encoding='utf-8')
(RUN_DIR/'10_training_manifest_audit.json').write_text(json.dumps(MANIFEST_REPORT,indent=2,default=str),encoding='utf-8')

(RUN_DIR/'10_normalization_contract_audit.json').write_text(json.dumps(SCALER_REPORTS,indent=2,default=str),encoding='utf-8')
(RUN_DIR/'10_checkpoint_architecture_audit.json').write_text(json.dumps(CHECKPOINT_REPORTS,indent=2,default=str),encoding='utf-8')
print('Normalizer audit:')
print(json.dumps(SCALER_REPORTS,indent=2,default=str))
print('\nDimension reconciliation:')
for label, cp in CHECKPOINT_REPORTS.items():
    print(label, 'checkpoint=',cp.get('input_dimension'),'->',cp.get('output_dimension'),
          '| Notebook06 source=',NB06_CONTRACT.get('input_dimension'),'->',NB06_CONTRACT.get('output_dimension'),
          '| matches=',cp.get('dimension_matches_notebook06_source'))
print('Per-model contract status:', {k:v['contract_status'] for k,v in MODEL_CONTRACT_REPORTS.items()})

Normalizer audit:
{
  "/content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1/06_training_normalization_v2.json": {
    "status": "READ",
    "path": "/content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1/06_training_normalization_v2.json",
    "top_level_type": "dict",
    "top_level_keys": [
      "seed",
      "input_columns",
      "output_columns",
      "x_mean",
      "x_std",
      "y_mean",
      "y_std",
      "normalization_split"
    ],
    "input_columns": {
      "source_key": "input_columns",
      "value": [
        "cell_voltage_V",
        "cell_current_A",
        "cell_temperature_K",
        "surface_temperature_K",
        "soc_proxy",
        "requested_charge_rate_C",
        "ambient_temperature_C",
        "horizon_s"
      ]
    },
    "output_columns": {
      "source_key": "output_columns",
      "value": [
        "capacity_retention_soh__future"
      ]
    },
    "x_mean": {
      "source_key": "x_mean",
      "value": [
        3.8397042751312256

## 6. Temporal schema, fault-onset provenance, and generator chronology

The CSV header is read without loading data. A bounded sample checks numeric time values and duplicate keys. The generator source is searched for explicit time-dependent fault injection. A `fault_cells_json`/`state_kind` label alone is static identity/state labeling—not an onset timestamp.

In [6]:
TS_PATH=ARTIFACTS.get('cell_timeseries')
SCENARIO_PATH=ARTIFACTS.get('scenario_instances')
ONSET_REPORT={
 'status':'NOT_EVALUABLE_MISSING_FAULT_ONSET_TIMESTAMP',
 'time_series_path':str(TS_PATH) if TS_PATH else None,
 'schema_status':'NOT_EVALUABLE',
 'onset_candidate_columns':[],
 'explicit_onset_evidence_found':False,
 'generator_source_status':SOURCE_REPORT.get('04',{}).get('status'),
 'interpretation':'No onset time is inferred from labels, scores, or first detector firing.'
}
TS_COLUMNS=[]
if TS_PATH and TS_PATH.is_file():
    try:
        hdr=pd.read_csv(TS_PATH,nrows=0)
        TS_COLUMNS=list(hdr.columns)
        time_candidates=[c for c in TS_COLUMNS if c.lower() in {'time_s','time_sec','timestamp_s','time','timestamp'}]
        onset_candidates=[c for c in TS_COLUMNS if any(t in c.lower() for t in ['fault_onset','onset_time','fault_start','injection_start','failure_time','event_onset','fault_start_time'])]
        ONSET_REPORT.update({'schema_status':'HEADER_READ','columns':TS_COLUMNS,'time_candidates':time_candidates,
                             'time_column':'time_s' if 'time_s' in TS_COLUMNS else (time_candidates[0] if len(time_candidates)==1 else None),
                             'onset_candidate_columns':onset_candidates,
                             'candidate_column_is_not_semantic_validation':True})
        if len(time_candidates)==1:
            ONSET_REPORT['time_schema_status']='PASS_HEADER_HAS_SINGLE_TIME_COLUMN'
        else:
            ONSET_REPORT['time_schema_status']='NOT_EVALUABLE_AMBIGUOUS_OR_MISSING_TIME_COLUMN'
    except Exception as e:
        ONSET_REPORT.update(schema_status='HEADER_READ_FAILED',error=f'{type(e).__name__}: {e}')
else:
    ONSET_REPORT['reason']='cell_timeseries.csv missing or ambiguous in Drive.'

# Search source code for explicit time-varying fault events, not generic fault labels.
source_hits=[]
for nbkey, text in SOURCE_TEXT.items():
    if nbkey != '04':
        continue  # inspect generator code only; audit notebooks mention onset fields as search strings
    lines=text.splitlines()
    for i,line in enumerate(lines):
        low=line.lower()
        if any(term in low for term in ['fault_onset_time','fault_start_time','injection_start_time','fault_start_s','onset_time_s','fault_injection_time']):
            source_hits.append({'notebook':nbkey,'line_number':i+1,'line':line.strip()[:300],
                                'context':'\n'.join(lines[max(0,i-2):min(len(lines),i+3)])[:1000]})
ONSET_REPORT['explicit_time_dependent_injection_source_hits']=source_hits
ONSET_REPORT['source_contains_static_fault_cell_assignment']=any(x['found'] for x in GENERATOR_EVIDENCE if x['pattern']=='is_fault = cell_id in fault_set')
ONSET_REPORT['source_contains_aged_healthy_state_assignment']=any(x['found'] for x in GENERATOR_EVIDENCE if x['pattern']=='state_kind = "aged" if is_fault else "healthy"')
if source_hits:
    ONSET_REPORT['status']='POTENTIAL_ONSET_SOURCE_REQUIRES_MANUAL_SEMANTIC_VALIDATION'
    ONSET_REPORT['explicit_onset_evidence_found']=True
else:
    ONSET_REPORT['status']='NOT_EVALUABLE_MISSING_FAULT_ONSET_TIMESTAMP'
    ONSET_REPORT['interpretation']='Available generator source assigns a fixed healthy/aged trajectory by fault cell for the whole simulated trajectory; no explicit event time was found in the inspected source. This supports missing onset ground truth for the current dataset, not proof that no other uninspected artifact could contain it.'

# Bounded sample only: no full-table load.
TEMPORAL_SAMPLE_REPORT={'status':'NOT_EVALUABLE'}
if TS_PATH and TS_PATH.is_file() and 'time_s' in TS_COLUMNS:
    sample_cols=[c for c in ['scenario_instance_id','scenario_id','cell_id','time_s','state_kind','fault_cells_json'] if c in TS_COLUMNS]
    try:
        sample=pd.read_csv(TS_PATH,usecols=sample_cols,nrows=50000,low_memory=False)
        sample['time_s']=pd.to_numeric(sample['time_s'],errors='coerce')
        TEMPORAL_SAMPLE_REPORT={'status':'BOUNDED_SAMPLE_ONLY','rows_read':int(len(sample)),'columns_read':sample_cols,
          'time_missing_count':int(sample['time_s'].isna().sum()),'time_min':float(sample['time_s'].min()) if sample['time_s'].notna().any() else None,
          'time_max':float(sample['time_s'].max()) if sample['time_s'].notna().any() else None,
          'duplicate_key_count_in_sample':int(sample.duplicated([c for c in ['scenario_instance_id','cell_id','time_s'] if c in sample_cols]).sum()) if all(c in sample_cols for c in ['scenario_instance_id','cell_id','time_s']) else None,
          'full_dataset_cleanliness_claim':'NOT_MADE_SAMPLE_IS_NOT_FULL_DATASET_VALIDATION'}
    except Exception as e:
        TEMPORAL_SAMPLE_REPORT={'status':'SAMPLE_AUDIT_FAILED','error':f'{type(e).__name__}: {e}'}

(RUN_DIR/'10_fault_onset_ground_truth_audit.json').write_text(json.dumps(ONSET_REPORT,indent=2,default=str),encoding='utf-8')
(RUN_DIR/'10_temporal_sample_audit.json').write_text(json.dumps(TEMPORAL_SAMPLE_REPORT,indent=2,default=str),encoding='utf-8')
print(json.dumps(ONSET_REPORT,indent=2,default=str))
print('\nBounded temporal sample:')
print(json.dumps(TEMPORAL_SAMPLE_REPORT,indent=2,default=str))

{
  "status": "NOT_EVALUABLE_MISSING_FAULT_ONSET_TIMESTAMP",
  "time_series_path": "/content/drive/MyDrive/PINN_EV_Battery/data/v1/cell_timeseries.csv",
  "schema_status": "HEADER_READ",
  "onset_candidate_columns": [],
  "explicit_onset_evidence_found": false,
  "generator_source_status": "NOT_EVALUABLE_SOURCE_MISSING_OR_AMBIGUOUS",
  "interpretation": "Available generator source assigns a fixed healthy/aged trajectory by fault cell for the whole simulated trajectory; no explicit event time was found in the inspected source. This supports missing onset ground truth for the current dataset, not proof that no other uninspected artifact could contain it.",
  "columns": [
    "scenario_id",
    "scenario_instance_id",
    "data_split",
    "pack_topology",
    "cell_id",
    "state_kind",
    "ageing_cycles",
    "ambient_temperature_C",
    "requested_charge_rate_C",
    "conditioning_duration_s",
    "time_s",
    "cell_voltage_V",
    "cell_current_A",
    "cell_temperature_K",
    "su

## 7. Re-evaluate the frozen Notebook 08 detector by family split

The threshold is read from the saved Notebook 08 summary when available. If that summary is missing, the reported frozen value `4.399034460052955` may be used for descriptive reproduction only, clearly labeled as not verified from the Drive artifact. The threshold is never recalibrated in this notebook.

Notebook 08's saved split manifest may contain only `family_key` and `evaluation_split`, not scenario IDs. In that case, this notebook reconstructs family keys using the exact Notebook 08 source rule and joins the frozen family assignments. It validates one-to-many/many-to-one keys and cell coverage before calculating metrics.

In [7]:
REPORTED_FROZEN_THRESHOLD=4.399034460052955
THRESHOLD_REPORT={'threshold':REPORTED_FROZEN_THRESHOLD,'source_status':'REPORTED_VALUE_FALLBACK_NOT_VERIFIED_FROM_DRIVE',
                  'frozen':True,'recalibrated_here':False}
summary_path=ARTIFACTS.get('notebook08_threshold_summary')
summary_obj,summary_read=read_json_safe(summary_path)
if isinstance(summary_obj,dict):
    tr=summary_obj.get('threshold_calibration',{})
    if isinstance(tr,dict) and isinstance(tr.get('threshold'),(int,float)):
        THRESHOLD_REPORT={'threshold':float(tr['threshold']),'source_status':'READ_FROM_NOTEBOOK08_SAVED_SUMMARY',
                          'summary_path':str(summary_path),'threshold_calibration':tr,'frozen':True,'recalibrated_here':False}
    else:
        THRESHOLD_REPORT.update({'summary_path':str(summary_path),'summary_status':'READ_BUT_THRESHOLD_FIELD_NOT_FOUND'})

SPLIT_PATH=ARTIFACTS.get('notebook08_split_manifest')
SCORES_PATH=ARTIFACTS.get('signal_scores')
FAMILY_AUDIT={'status':'NOT_EVALUABLE_MISSING_REQUIRED_ARTIFACTS','threshold':THRESHOLD_REPORT}
METRICS_DF=None; SCENARIO_METRICS_DF=None; JOINED_SCORES=None

def choose_col(df, options):
    return next((x for x in options if x in df.columns),None)

def parse_fault_cells(value):
    if value is None or (isinstance(value,float) and np.isnan(value)): return set()
    if isinstance(value,(list,tuple,set,np.ndarray)): raw=list(value)
    else:
        st=str(value).strip()
        if st.lower() in {'','nan','none','null'}: return set()
        try: raw=json.loads(st)
        except Exception as e: raise ValueError(f'fault cell metadata is not valid JSON: {st[:100]}') from e
    out=set()
    for x in raw or []:
        if x is None: continue
        cell=int(x)
        if not 1<=cell<=100: raise ValueError(f'fault cell {cell} outside 1..100')
        out.add(cell)
    return out

if all(p and Path(p).is_file() for p in [SCENARIO_PATH,SCORES_PATH,SPLIT_PATH]):
  try:
    meta=pd.read_csv(SCENARIO_PATH,low_memory=False)
    scores=pd.read_csv(SCORES_PATH,low_memory=False)
    split_manifest=pd.read_csv(SPLIT_PATH,low_memory=False)
    idcol=choose_col(meta,['scenario_instance_id','instance_id'])
    scencol=choose_col(meta,['scenario_id'])
    faultcol=choose_col(meta,['fault_cells_json','fault_cells'])
    scoreid=choose_col(scores,['scenario_instance_id','instance_id'])
    cellcol=choose_col(scores,['cell_id'])
    scorecol=choose_col(scores,['signal_anomaly_score'])
    famcol=choose_col(split_manifest,['family_key','family_id'])
    splitcol=choose_col(split_manifest,['evaluation_split','split','data_split'])
    required={'metadata id':idcol,'metadata scenario':scencol,'metadata fault cells':faultcol,
              'score scenario id':scoreid,'score cell id':cellcol,'score value':scorecol,
              'manifest family':famcol,'manifest split':splitcol}
    missing=[k for k,v in required.items() if v is None]
    if missing: raise ValueError('Required join columns missing: '+str(missing))
    meta=meta.copy(); scores=scores.copy(); split_manifest=split_manifest.copy()
    meta[idcol]=meta[idcol].astype(str).str.strip(); meta[scencol]=meta[scencol].astype(str).str.strip()
    scores[scoreid]=scores[scoreid].astype(str).str.strip()
    if meta[idcol].duplicated().any(): raise ValueError('Duplicate scenario IDs in metadata; refusing ambiguous join.')
    scores[cellcol]=pd.to_numeric(scores[cellcol],errors='coerce')
    scores[scorecol]=pd.to_numeric(scores[scorecol],errors='coerce')
    if scores[[cellcol,scorecol]].isna().any().any(): raise ValueError('Non-numeric/missing cell ID or score.')
    if not np.isfinite(scores[scorecol].to_numpy(float)).all(): raise ValueError('Non-finite signal scores.')
    scores[cellcol]=scores[cellcol].astype(int)
    if scores.duplicated([scoreid,cellcol]).any(): raise ValueError('Duplicate score rows per scenario/cell; refusing join.')
    if not scores[cellcol].between(1,100).all(): raise ValueError('Cell IDs outside 1..100.')
    meta['fault_cells_parsed']=meta[faultcol].apply(parse_fault_cells)
    meta['has_fault']=meta['fault_cells_parsed'].apply(bool)
    meta['family_key']=meta.apply(lambda r:f"SCENARIO::{r[scencol]}::FAULTS::{','.join(map(str,sorted(r['fault_cells_parsed'])))}",axis=1)
    basecol=choose_col(meta,['base_scenario_instance_id'])
    if basecol:
        id_to_family=dict(zip(meta[idcol],meta['family_key']))
        for ix,v in meta[basecol].items():
            if pd.isna(v) or str(v).strip().lower() in {'','nan','none','null'}: continue
            baseid=str(v).strip()
            meta.at[ix,'family_key']=id_to_family.get(baseid,f'BASE::{baseid}')
    split_manifest=split_manifest[[famcol,splitcol]].copy()
    split_manifest[famcol]=split_manifest[famcol].astype(str).str.strip()
    split_manifest[splitcol]=split_manifest[splitcol].astype(str).str.strip().str.lower().replace({'validation':'val'})
    if split_manifest[famcol].duplicated().any(): raise ValueError('Family split manifest has duplicate family keys with potentially conflicting assignments.')
    if not set(split_manifest[splitcol]).issubset({'train','val','test'}): raise ValueError('Unexpected split values in frozen manifest.')
    if meta['family_key'].isna().any(): raise ValueError('Family reconstruction produced missing keys.')
    meta=meta.merge(split_manifest.rename(columns={famcol:'family_key',splitcol:'evaluation_split'}),on='family_key',how='left',validate='many_to_one',indicator=True)
    missing_families=meta.loc[meta['_merge']!='both','family_key'].unique().tolist()
    if missing_families: raise ValueError(f'Metadata families missing from frozen split manifest: {missing_families[:20]}')
    meta=meta.drop(columns=['_merge'])
    score_scenarios=set(scores[scoreid]); metadata_scenarios=set(meta[idcol])
    missing_meta=sorted(score_scenarios-metadata_scenarios)
    unscored=sorted(metadata_scenarios-score_scenarios)
    if missing_meta: raise ValueError(f'Scores reference scenarios absent from metadata: {missing_meta[:20]}')
    score_coverage=scores.groupby(scoreid)[cellcol].agg(lambda x:sorted(set(x)))
    bad_coverage={str(k):len(v) for k,v in score_coverage.items() if v!=list(range(1,101))}
    # Validate redundant scenario_id in the score file against metadata, then drop it before merge to avoid suffix ambiguity.
    if scencol in scores.columns:
        expected_scenario=meta.set_index(idcol)[scencol].to_dict()
        mismatch_rows=[(sid,str(sval),str(expected_scenario.get(str(sid)))) for sid,sval in zip(scores[scoreid],scores[scencol])
                       if str(sval).strip()!=str(expected_scenario.get(str(sid))).strip()]
        if mismatch_rows: raise ValueError(f'Score scenario_id disagrees with metadata; examples: {mismatch_rows[:10]}')
        scores=scores.drop(columns=[scencol])
    joined=scores.merge(meta[[idcol,scencol,'fault_cells_parsed','has_fault','family_key','evaluation_split']],
                        left_on=scoreid,right_on=idcol,how='left',validate='many_to_one',indicator=True)
    if (joined['_merge']!='both').any(): raise ValueError('Unmatched score rows remain after validated join.')
    joined['is_fault']=[int(int(c) in f) for c,f in zip(joined[cellcol],joined['fault_cells_parsed'])]
    joined['predicted_fault']=joined[scorecol]>=float(THRESHOLD_REPORT['threshold'])
    # Compare any precomputed label in score file with labels derived from scenario metadata.
    labelcol=choose_col(scores,['is_fault','fault_label','actual_fault'])
    label_mismatch_count=None
    if labelcol:
        existing=pd.to_numeric(joined[labelcol],errors='coerce')
        label_mismatch_count=int((existing.notna() & (existing.astype('Int64')!=joined['is_fault'].astype('Int64'))).sum())
        if label_mismatch_count: raise ValueError(f'Pre-existing labels conflict with metadata-derived fault labels: {label_mismatch_count}')
    split_sets={s:set(meta.loc[meta['evaluation_split']==s,'family_key']) for s in ['train','val','test']}
    overlaps={f'{a}_{b}':sorted(split_sets[a]&split_sets[b]) for a,b in [('train','val'),('train','test'),('val','test')]}
    if any(overlaps.values()): raise ValueError(f'Family overlap in frozen split: {overlaps}')
    def calc_metrics(df, splitname):
        y=df['is_fault'].to_numpy(bool); pred=df['predicted_fault'].to_numpy(bool)
        tp=int((pred&y).sum()); fp=int((pred&~y).sum()); fn=int((~pred&y).sum()); tn=int((~pred&~y).sum())
        def div(a,b): return float(a/b) if b else None
        prec=div(tp,tp+fp); rec=div(tp,tp+fn); spec=div(tn,tn+fp)
        f1=(2*prec*rec/(prec+rec)) if prec is not None and rec is not None and prec+rec>0 else (0.0 if prec==0 and rec==0 else None)
        return {'split':splitname,'status':'CALCULATED_DESCRIPTIVE_STATIC_LABEL_DETECTION',
          'families':int(df['family_key'].nunique()),'scenarios':int(df[idcol].nunique()),
          'healthy_scenarios':int(df.loc[~df['has_fault'],idcol].nunique()),'fault_scenarios':int(df.loc[df['has_fault'],idcol].nunique()),
          'cell_rows':int(len(df)),'positive_cells':int(y.sum()),'negative_cells':int((~y).sum()),
          'TP':tp,'FP':fp,'FN':fn,'TN':tn,'precision':prec,'recall':rec,'specificity':spec,'F1':f1,
          'class_support_valid':bool(y.any() and (~y).any()),'independent_family_count':int(df['family_key'].nunique())}
    rows=[]
    for splitname in ['train','val','test']:
        part=joined[joined['evaluation_split']==splitname]
        if len(part): rows.append(calc_metrics(part,splitname))
        else: rows.append({'split':splitname,'status':'NOT_EVALUABLE_NO_ROWS'})
    METRICS_DF=pd.DataFrame(rows)
    scen_rows=[]
    for sid,g in joined.groupby(idcol,sort=True):
        y=g['is_fault'].to_numpy(bool); pred=g['predicted_fault'].to_numpy(bool)
        tp=int((y&pred).sum()); fp=int((~y&pred).sum()); fn=int((y&~pred).sum()); tn=int((~y&~pred).sum())
        scen_rows.append({'scenario_instance_id':str(sid),'scenario_id':str(g[scencol].iloc[0]),'family_key':str(g['family_key'].iloc[0]),
          'split':str(g['evaluation_split'].iloc[0]),'cells_scored':int(len(g)),'actual_fault_cells':int(y.sum()),
          'predicted_fault_cells':int(pred.sum()),'TP':tp,'FP':fp,'FN':fn,'TN':tn,
          'healthy_scenario_false_alarm':bool(not y.any() and pred.any()),'exact_fault_cell_set_match':bool(np.array_equal(y,pred)),
          'threshold':float(THRESHOLD_REPORT['threshold'])})
    SCENARIO_METRICS_DF=pd.DataFrame(scen_rows)
    JOINED_SCORES=joined
    full_coverage_ok=(not bad_coverage and not unscored and not missing_meta and (label_mismatch_count in [None,0]))
    split_ok=not any(overlaps.values())
    FAMILY_AUDIT={'status':'VALIDATED_JOIN_AND_SPLIT_AUDIT' if full_coverage_ok and split_ok else 'JOIN_VALID_WITH_COVERAGE_OR_SPLIT_ISSUES',
      'metadata_rows':int(len(meta)),'score_rows':int(len(scores)),
      'metadata_scenarios':int(meta[idcol].nunique()),'scored_scenarios':int(scores[scoreid].nunique()),
      'unscored_metadata_scenarios':unscored,'missing_metadata_scenarios_for_scores':missing_meta,
      'duplicate_metadata_ids':False,'duplicate_score_scenario_cell_keys':False,'bad_100_cell_coverage_scenarios':bad_coverage,
      'full_scenario_coverage_pass':full_coverage_ok,'preexisting_label_column':labelcol,'label_mismatch_count':label_mismatch_count,
      'manifest_columns':list(pd.read_csv(SPLIT_PATH,nrows=0).columns),'manifest_row_count':int(len(split_manifest)),
      'family_key_reconstruction_rule':'exact Notebook 08 default family SCENARIO::<scenario_id>::FAULTS::<sorted comma-separated fault cells>; base_scenario_instance_id overrides via Notebook 08 one-pass id_to_family mapping',
      'family_split_overlap':overlaps,'family_isolation_pass':split_ok,
      'threshold_report':THRESHOLD_REPORT,'metrics_are_static_label_detection_not_early_warning':True}
  except Exception as e:
    FAMILY_AUDIT={'status':'AUDIT_FAILED_OR_NOT_EVALUABLE','error':f'{type(e).__name__}: {e}',
      'traceback':traceback.format_exc(limit=5),'threshold_report':THRESHOLD_REPORT}
else:
  FAMILY_AUDIT={'status':'NOT_EVALUABLE_MISSING_REQUIRED_ARTIFACTS',
    'missing':[str(p) for p in [SCENARIO_PATH,SCORES_PATH,SPLIT_PATH] if not p or not Path(p).is_file()],
    'threshold_report':THRESHOLD_REPORT}

if METRICS_DF is not None:
    METRICS_DF.to_csv(RUN_DIR/'10_detector_metrics_by_split.csv',index=False)
if SCENARIO_METRICS_DF is not None:
    SCENARIO_METRICS_DF.to_csv(RUN_DIR/'10_detector_scenario_metrics.csv',index=False)
(RUN_DIR/'10_family_split_and_join_audit.json').write_text(json.dumps(FAMILY_AUDIT,indent=2,default=str),encoding='utf-8')
(RUN_DIR/'10_threshold_provenance.json').write_text(json.dumps(THRESHOLD_REPORT,indent=2,default=str),encoding='utf-8')
print(json.dumps(FAMILY_AUDIT,indent=2,default=str))
if METRICS_DF is not None: display(METRICS_DF)
if SCENARIO_METRICS_DF is not None: display(SCENARIO_METRICS_DF.head(30))

{
  "status": "VALIDATED_JOIN_AND_SPLIT_AUDIT",
  "metadata_rows": 28,
  "score_rows": 2800,
  "metadata_scenarios": 28,
  "scored_scenarios": 28,
  "unscored_metadata_scenarios": [],
  "missing_metadata_scenarios_for_scores": [],
  "duplicate_metadata_ids": false,
  "duplicate_score_scenario_cell_keys": false,
  "bad_100_cell_coverage_scenarios": {},
  "full_scenario_coverage_pass": true,
  "preexisting_label_column": null,
  "label_mismatch_count": null,
  "manifest_columns": [
    "family_key",
    "evaluation_split"
  ],
  "manifest_row_count": 13,
  "family_key_reconstruction_rule": "exact Notebook 08 default family SCENARIO::<scenario_id>::FAULTS::<sorted comma-separated fault cells>; base_scenario_instance_id overrides via Notebook 08 one-pass id_to_family mapping",
  "family_split_overlap": {
    "train_val": [],
    "train_test": [],
    "val_test": []
  },
  "family_isolation_pass": true,
  "threshold_report": {
    "threshold": 4.399034460052955,
    "source_status": "READ_F

,split,status,families,scenarios,healthy_scenarios,fault_scenarios,cell_rows,positive_cells,negative_cells,TP,FP,FN,TN,precision,recall,specificity,F1,class_support_valid,independent_family_count
0,train,CALCULATED_DESCRIPTIVE_STATIC_LABEL_DETECTION,7,17,1,16,1700,22,1678,22,0,0,1678,1.0,1.0,1.0,1.0,True,7
1,val,CALCULATED_DESCRIPTIVE_STATIC_LABEL_DETECTION,3,3,1,2,300,2,298,2,0,0,298,1.0,1.0,1.0,1.0,True,3
2,test,CALCULATED_DESCRIPTIVE_STATIC_LABEL_DETECTION,3,8,1,7,800,7,793,7,0,0,793,1.0,1.0,1.0,1.0,True,3


,scenario_instance_id,scenario_id,family_key,split,cells_scored,actual_fault_cells,predicted_fault_cells,TP,FP,FN,TN,healthy_scenario_false_alarm,exact_fault_cell_set_match,threshold
0,S1_HEALTHY,S1,SCENARIO::S1::FAULTS::,train,100,0,0,0,0,0,100,False,True,4.399034
1,S2_HEALTHY,S2,SCENARIO::S2::FAULTS::,val,100,0,0,0,0,0,100,False,True,4.399034
2,S3_HEALTHY,S3,SCENARIO::S3::FAULTS::,test,100,0,0,0,0,0,100,False,True,4.399034
3,S4_CELL004,S4,SCENARIO::S4::FAULTS::4,train,100,1,1,1,0,0,99,False,True,4.399034
4,S4_CELL019,S4,SCENARIO::S4::FAULTS::19,test,100,1,1,1,0,0,99,False,True,4.399034
5,S4_CELL037_BENCH,S4,SCENARIO::S4::FAULTS::37,train,100,1,1,1,0,0,99,False,True,4.399034
6,S4_CELL055,S4,SCENARIO::S4::FAULTS::55,val,100,1,1,1,0,0,99,False,True,4.399034
7,S4_CELL082,S4,SCENARIO::S4::FAULTS::82,train,100,1,1,1,0,0,99,False,True,4.399034
8,S5_CELL004,S5,SCENARIO::S5::FAULTS::4,train,100,1,1,1,0,0,99,False,True,4.399034
9,S5_CELL004_082,S5,"SCENARIO::S5::FAULTS::4,82",train,100,2,2,2,0,0,98,False,True,4.399034


## 8. Evidence ledger and correction of Notebook 09 claims

The ledger separates directly inspected source from claims that were only reported in the conversation. Notebook 09's printed status is not accepted automatically. Runtime artifacts can strengthen, weaken, or leave each finding unresolved.

In [8]:
ledger=[]
def add_evidence(category,claim,evidence,location,corrected,implication,status='UNSUPPORTED'):
    ledger.append({'category':category,'original_claim_or_status':claim,'evidence_used':evidence,
                   'exact_file_or_check':location,'corrected_conclusion':corrected,'implication_for_next_step':implication,'status':status})

add_evidence('VERIFIED','Notebook 06 uses a multi-output future-target architecture',
 'Read-only AST/source inspection extracted FEATURE_COLS, TARGET_COLS, INPUT_COLS=FEATURE_COLS+[horizon_s], PredictiveMLP width/depth and scaler code.',
 str(NOTEBOOKS.get('06',[None])[0]) if len(NOTEBOOKS.get('06',[]))==1 else 'Notebook 06 source unavailable',
 f"Source contract extracted: {NB06_CONTRACT.get('input_dimension')} inputs and {NB06_CONTRACT.get('output_dimension')} future targets; targets include multiple state variables, not SOH-only.",
 'Only use checkpoints/scalers proven to match this source contract; 60/300/600-second SOH-only assumptions are not the same model contract.',
 'VERIFIED' if NB06_CONTRACT.get('status')=='SOURCE_CONTRACT_EXTRACTED' else 'NOT_EVALUABLE')

for label,cp in CHECKPOINT_REPORTS.items():
    status='VERIFIED' if cp.get('status')=='SAFE_STATE_DICT_INSPECTED' else 'NOT_EVALUABLE'
    add_evidence('MODEL_CONTRACT',f'{label} is a valid future-SOH checkpoint',
      f"Safe checkpoint inspection status={cp.get('status')}; dimensions={cp.get('input_dimension')}->{cp.get('output_dimension')}; source contract={NB06_CONTRACT.get('input_dimension')}->{NB06_CONTRACT.get('output_dimension')}.",
      cp.get('path',cp.get('path_candidates','missing')),
      'Inference remains disabled until checkpoint, ordered feature metadata, target semantics, and scaler are all compatible.',
      'Resolve from exact training manifest/scaler and the source checkpoint actually produced by Notebook 06.',status)

add_evidence('NORMALIZATION','normalization_split=train_only proves valid normalization',
 'Normalizer fields and array lengths inspected; a split string is recorded as a claim, not independent proof of scaler provenance.',
 '; '.join(SCALER_REPORTS.keys()) or 'No scaler JSON found',
 'Train-only provenance is not verified unless the scaler matches the checkpoint/source and training artifact; ordered features and target scales must match.',
 'Use matching saved scaler and manifest; do not force dimensions.',
 'VERIFIED' if SCALER_REPORTS else 'NOT_EVALUABLE')

add_evidence('FAULT_ONSET','Fault onset ground truth available',
 'Time-series header/sample inspected; Notebook 04 source inspected for fixed fault-cell-to-aged-trajectory assignment and explicit time-dependent onset fields.',
 str(TS_PATH) if TS_PATH else 'cell_timeseries.csv unavailable',
 ONSET_REPORT.get('status')+'; no onset time is inferred.',
 'Generate a separate versioned temporal fault-injection dataset with explicit injection timestamps and documented mechanism.',
 'NOT_EVALUABLE' if ONSET_REPORT.get('status','').startswith('NOT_EVALUABLE') else 'UNSUPPORTED')

add_evidence('SPLIT','Notebook 08 family split manifest is enough by itself to prove test isolation',
 'Manifest schema and family-key reconstruction checked; split overlaps and joins validated where files exist.',
 str(SPLIT_PATH) if SPLIT_PATH else 'Notebook 08 split manifest unavailable',
 FAMILY_AUDIT.get('status','NOT_EVALUABLE')+'; no independent test claim unless exact family join and zero overlap pass.',
 'Report split-specific static detector metrics only if all join and denominator checks pass.',
 'VERIFIED' if FAMILY_AUDIT.get('status')=='VALIDATED_JOIN_AND_SPLIT_AUDIT' else 'NOT_EVALUABLE')

add_evidence('DETECTOR_METRICS','Pooled 100% metrics prove early-warning performance',
 'Notebook 09 reported TP=31, FP=0, FN=0, TN=2769 over 2800 rows; Notebook 08 source labels the method a signal-statistics detector with static fault-cell labels.',
 'Notebook 09 reported outputs plus Notebook 08 source cells for score-label join and metrics',
 'Pooled cell-level descriptive metrics do not establish independent test performance or pre-onset warning. This notebook recomputes train/val/test metrics from the frozen family split when artifacts are available.',
 'Keep detector evaluation separate from PINN inference and lead-time evaluation.',
 'VERIFIED')

add_evidence('TEMPORAL_SCHEMA','time_s existing proves onset timestamps exist',
 f"Header time column status={ONSET_REPORT.get('time_schema_status')}; onset candidates={ONSET_REPORT.get('onset_candidate_columns')}.",
 str(TS_PATH) if TS_PATH else 'cell_timeseries.csv unavailable',
 'A trajectory time coordinate is not an event-onset label.',
 'Lead-time metrics remain NOT_EVALUABLE without semantic onset ground truth.',
 'VERIFIED' if TS_PATH and ONSET_REPORT.get('time_schema_status')=='PASS_HEADER_HAS_SINGLE_TIME_COLUMN' else 'NOT_EVALUABLE')

add_evidence('REPRODUCIBILITY','Notebook 09 research gates are all validated because the notebook completed',
 'Notebook 09 source marks some gates from artifact existence/array shape and hard-codes onset unavailable; Notebook 10 independently rechecks each relevant condition.',
 str(NOTEBOOKS.get('09',[None])[0]) if len(NOTEBOOKS.get('09',[]))==1 else 'Notebook 09 source unavailable',
 'Engineering completion and scientific validity are separate; each gate below uses evidence-backed status.',
 'Do not infer test-only usage, no leakage, or real-world safety from successful execution.',
 'VERIFIED' if len(NOTEBOOKS.get('09',[]))==1 else 'NOT_EVALUABLE')

ledger_df=pd.DataFrame(ledger)
ledger_df.to_csv(RUN_DIR/'10_evidence_ledger.csv',index=False)
print(ledger_df[['category','status','original_claim_or_status','corrected_conclusion']].to_string(index=False))

        category        status                                                      original_claim_or_status                                                                                                                                                                                                 corrected_conclusion
        VERIFIED NOT_EVALUABLE                    Notebook 06 uses a multi-output future-target architecture                                                                                              Source contract extracted: None inputs and None future targets; targets include multiple state variables, not SOH-only.
  MODEL_CONTRACT      VERIFIED                           B1_legacy_reported is a valid future-SOH checkpoint                                                                                              Inference remains disabled until checkpoint, ordered feature metadata, target semantics, and scaler are all compatible.
  MODEL_CONTRACT      VERIFIED    

## 9. Conservative readiness gates and overall decision

A PASS applies only to the condition explicitly tested. Static fault classification, future-SOH prediction, cell localization, and pre-onset lead time are separate research claims.

In [9]:
def gate_status(condition, true_status='PASS', false_status='FAIL', unknown_status='NOT_EVALUABLE'):
    if condition is True: return true_status
    if condition is False: return false_status
    return unknown_status

project_found=bool(PROJECT_ROOT.is_dir() and ARTIFACTS.get('scenario_instances') and len(NOTEBOOKS.get('06',[]))==1)
source_contract_ok=(NB06_CONTRACT.get('status')=='SOURCE_CONTRACT_EXTRACTED' and NB06_CONTRACT.get('input_dimension') and NB06_CONTRACT.get('output_dimension'))
checkpoint_verified_any=any(cp.get('status')=='SAFE_STATE_DICT_INSPECTED' for cp in CHECKPOINT_REPORTS.values())
checkpoint_contract_match_any=any(cp.get('dimension_matches_notebook06_source') is True for cp in CHECKPOINT_REPORTS.values())
# Semantic normalization validation intentionally requires exact ordered columns, matching array lengths, and source correspondence.
normalization_semantic_pass=bool(
    'scaler_order_matches' in globals() and scaler_order_matches and scaler_arrays_ok and scaler_finite_ok and scaler_scales_positive and
    nb06_train_only_source and manifest_contract_matches and manifest_horizons_match
  )

temporal_schema_pass=ONSET_REPORT.get('time_schema_status')=='PASS_HEADER_HAS_SINGLE_TIME_COLUMN'
onset_pass=False # no source-backed semantic onset definition established in this audit
family_pass=(True if FAMILY_AUDIT.get('status')=='VALIDATED_JOIN_AND_SPLIT_AUDIT' and FAMILY_AUDIT.get('family_isolation_pass') is True else (False if FAMILY_AUDIT.get('status')=='JOIN_VALID_WITH_COVERAGE_OR_SPLIT_ISSUES' else None))
threshold_verified=THRESHOLD_REPORT.get('source_status')=='READ_FROM_NOTEBOOK08_SAVED_SUMMARY'
metrics_available=METRICS_DF is not None and not METRICS_DF.empty
if metrics_available and bool((METRICS_DF['split']=='test').any()) and family_pass is not None:
    test_row=METRICS_DF.loc[METRICS_DF['split']=='test'].iloc[0]
    independent_test_eval=bool(family_pass and test_row.get('class_support_valid',False) and
      test_row.get('families',0)>=2 and test_row.get('healthy_scenarios',0)>=1 and test_row.get('fault_scenarios',0)>=1)
else:
    independent_test_eval=None

GATES={
 'project_artifacts_found':gate_status(project_found),
 'notebook06_source_contract_extracted':gate_status(bool(source_contract_ok),unknown_status='NOT_EVALUABLE'),
 'b1_inference_supported':('PASS' if MODEL_CONTRACT_REPORTS.get('B1',{}).get('inference_supported_for_smoke_test') else 'NOT_EVALUABLE'),
 'b3_inference_supported':('PASS' if MODEL_CONTRACT_REPORTS.get('B3',{}).get('inference_supported_for_smoke_test') else 'NOT_EVALUABLE'),
 'normalization_contract_validated':gate_status(bool(normalization_semantic_pass),unknown_status='NOT_EVALUABLE'),
 'future_soh_prediction_supported':('PASS' if any('capacity_retention_soh__future' in (x.get('ordered_output_targets') or []) and x.get('supported_horizons_seconds')==[60,300,600] and x.get('contract_status')=='CONTRACT_MATCHES_SOURCE_FOR_INFERENCE_SMOKE_TEST' for m in MODEL_CONTRACT_REPORTS.values() for x in m.get('candidate_checkpoints',[])) else 'NOT_EVALUABLE'),
 'temporal_data_schema_validated':gate_status(temporal_schema_pass),
 'fault_onset_ground_truth_available':gate_status(onset_pass,false_status='NOT_EVALUABLE'),
 'train_validation_test_family_isolation':gate_status(family_pass,unknown_status='NOT_EVALUABLE'),
 'no_ground_truth_feature_leakage':'NOT_EVALUABLE', # detector score-generation source for this exact score file was not proven label-independent
 'test_set_used_only_for_final_evaluation':'NOT_EVALUABLE', # historical test-use cannot be inferred from a split manifest alone
 'threshold_or_model_selection_train_validation_only':('PASS' if THRESHOLD_REPORT.get('source_status')=='READ_FROM_NOTEBOOK08_SAVED_SUMMARY' and THRESHOLD_REPORT.get('threshold_calibration',{}).get('threshold_calibrated_using_training_families_only') is True else 'NOT_EVALUABLE'),
 'threshold_provenance_verified':gate_status(threshold_verified,unknown_status='NOT_EVALUABLE'),
 'threshold_not_recalibrated_in_notebook10':'PASS' if THRESHOLD_REPORT.get('recalibrated_here') is False else 'FAIL',
 'independent_held_out_detector_evaluation_supported':gate_status(independent_test_eval,unknown_status='NOT_EVALUABLE'),
 'cell_localization_readiness':'NOT_EVALUABLE',
 'prediction_arrays_finite':'NOT_APPLICABLE_NO_PINN_INFERENCE_RUN',
 'metric_denominators_valid':(gate_status(bool(METRICS_DF['class_support_valid'].fillna(False).all()),unknown_status='NOT_EVALUABLE') if metrics_available else 'NOT_EVALUABLE'),
 'baseline_comparison_valid':'NOT_EVALUABLE_NO_MATCHED_VALID_PINN_INFERENCE',
 'lead_time_metrics_evaluable':gate_status(onset_pass,false_status='NOT_EVALUABLE'),
 'early_warning_performance_demonstrated':gate_status(False,false_status='NOT_EVALUABLE'),
 'new_temporal_fault_injection_dataset_required':'PASS' if not onset_pass else 'NOT_APPLICABLE',
 'reproducibility_artifacts_saved':'PASS'
}

if GATES['b1_inference_supported']=='PASS' and GATES['b3_inference_supported']=='PASS' and GATES['normalization_contract_validated']=='PASS' and GATES['fault_onset_ground_truth_available']=='PASS':
    overall='READY_FOR_NEXT_VALIDATION_STAGE'
elif not project_found or not source_contract_ok:
    overall='BLOCKED_MISSING_CRITICAL_EVIDENCE'
else:
    overall='PARTIALLY_READY_WITH_BLOCKERS'

READINESS={
 'overall_readiness':overall,
 'subgates':GATES,
 'b1_inference_supported':GATES['b1_inference_supported'],
 'b3_inference_supported':GATES['b3_inference_supported'],
 'future_soh_prediction_supported':GATES['future_soh_prediction_supported'],
 'held_out_signal_detector_evaluation_supported':GATES['independent_held_out_detector_evaluation_supported'],
 'early_warning_lead_time_evaluation_supported':GATES['lead_time_metrics_evaluable'],
 'new_temporal_fault_injection_dataset_required':not onset_pass,
 'model_contract_source':NB06_CONTRACT,
 'onset_status':ONSET_REPORT.get('status'),
 'threshold_status':THRESHOLD_REPORT.get('source_status'),
 'run_directory':str(RUN_DIR),
 'interpretation':'B1/B3 PASS means contract-compatible for a separate inference smoke test only; this notebook did not run inference. A PASS on static signal detection does not establish pre-onset warning or real-world safety.'
}
(RUN_DIR/'10_readiness_gate_summary.json').write_text(json.dumps(READINESS,indent=2,default=str),encoding='utf-8')
print(json.dumps(READINESS,indent=2,default=str))

{
  "overall_readiness": "BLOCKED_MISSING_CRITICAL_EVIDENCE",
  "subgates": {
    "project_artifacts_found": "FAIL",
    "notebook06_source_contract_extracted": "FAIL",
    "b1_inference_supported": "NOT_EVALUABLE",
    "b3_inference_supported": "NOT_EVALUABLE",
    "normalization_contract_validated": "FAIL",
    "future_soh_prediction_supported": "NOT_EVALUABLE",
    "temporal_data_schema_validated": "PASS",
    "fault_onset_ground_truth_available": "NOT_EVALUABLE",
    "train_validation_test_family_isolation": "PASS",
    "no_ground_truth_feature_leakage": "NOT_EVALUABLE",
    "test_set_used_only_for_final_evaluation": "NOT_EVALUABLE",
    "threshold_or_model_selection_train_validation_only": "PASS",
    "threshold_provenance_verified": "PASS",
    "threshold_not_recalibrated_in_notebook10": "PASS",
    "independent_held_out_detector_evaluation_supported": "PASS",
    "cell_localization_readiness": "NOT_EVALUABLE",
    "prediction_arrays_finite": "NOT_APPLICABLE_NO_PINN_INFERENCE_RUN

## 10. Final report and artifact manifest

The report lists missing evidence and actionable next steps. No PINN inference, model training, test-based threshold selection, or dataset modification occurs in Notebook 10.

In [10]:
missing_critical=[]
if not project_found: missing_critical.append('Google Drive project root or required artifacts were unavailable in this runtime.')
if not source_contract_ok: missing_critical.append('Notebook 06 source contract could not be extracted from the source notebook.')
if not normalization_semantic_pass: missing_critical.append('No scaler was proven to match exact ordered inputs/outputs and training provenance.')
if not any(cp.get('dimension_matches_notebook06_source') is True for cp in CHECKPOINT_REPORTS.values()): missing_critical.append('No checkpoint was proven dimension-compatible with the inspected Notebook 06 source contract.')
if not onset_pass: missing_critical.append('No semantically verified fault-onset timestamp/event definition is available.')
if FAMILY_AUDIT.get('status')!='VALIDATED_JOIN_AND_SPLIT_AUDIT': missing_critical.append('Frozen Notebook 08 score-to-family split join could not be fully validated.')
if not threshold_verified: missing_critical.append('Frozen threshold source could not be verified from Notebook 08 saved summary; only the reported fallback may be available.')

final_report={
 'notebook':'10_V1_ModelContract_and_FaultOnset_GroundTruth_FINAL',
 'overall_readiness':overall,
 'engineering_status':'AUDIT_NOTEBOOK_EXECUTED_TO_THIS_CELL' if RUN_DIR.exists() else 'NOT_EVALUABLE',
 'source_notebook_audit':SOURCE_REPORT,
 'notebook06_source_contract':NB06_CONTRACT,
 'checkpoint_reports':CHECKPOINT_REPORTS,
 'model_contract_reports':MODEL_CONTRACT_REPORTS,
 'training_manifest_audit':MANIFEST_REPORT,
 'normalizer_reports':SCALER_REPORTS,
 'fault_onset_ground_truth_audit':ONSET_REPORT,
 'temporal_sample_audit':TEMPORAL_SAMPLE_REPORT,
 'family_split_and_join_audit':FAMILY_AUDIT,
 'frozen_threshold_report':THRESHOLD_REPORT,
 'gates':GATES,
 'missing_critical_evidence':missing_critical,
 'scientific_conclusion':('Early-warning lead time is NOT EVALUABLE until onset ground truth is explicitly documented and independently checked. '
    'The accessible Notebook 04 source appears to assign a fixed aged/healthy trajectory to each cell for the entire simulated run, not to inject a fault at a documented time. '
    'Notebook 06 source describes a multi-output future-state model with 11 inputs and 7 outputs; this differs from the previously reported legacy 7-input checkpoint shape and an 8-input SOH-only scaler. '
    'No checkpoint inference is authorized unless a checkpoint, ordered features, target semantics, and scaler are all proven to match.'),
 'next_actions':[
    'Use the exact Notebook 06 outputs under data/v1/models_v1 (checkpoint plus v1_scaler.json and 06_training_manifest.json) only if their architecture and ordered contract match the source.',
    'Resolve legacy results_v1 checkpoints separately; do not coerce 7-input weights to an 11-input model or an 8-column normalizer.',
    'Build a new versioned temporal fault-injection dataset with explicit per-scenario/cell fault mechanism, injection_time_s, event semantics, and a pre-onset baseline; do not alter frozen v1 data.',
    'Re-evaluate frozen Notebook 08 threshold on the saved family split; report per-split and scenario metrics with family counts and denominators.',
    'Only after contracts and onset labels pass, design a preregistered warning window, event-level metrics, false alarms per cell-hour/trajectory, and lead-time evaluation on untouched test families.'
 ],
 'run_directory':str(RUN_DIR)
}
(RUN_DIR/'10_final_audit_report.json').write_text(json.dumps(final_report,indent=2,default=str),encoding='utf-8')

lines=[
 '# Notebook 10 — Final Model Contract & Fault-Onset Audit', '',
 f'- **Overall readiness:** `{overall}`',
 f'- **Run directory:** `{RUN_DIR}`',
 '', '## Key conclusions',
 f"- Notebook 06 source contract status: `{NB06_CONTRACT.get('status')}`; input/output dimensions: `{NB06_CONTRACT.get('input_dimension')} -> {NB06_CONTRACT.get('output_dimension')}`.",
 f"- B1 model contract: `{MODEL_CONTRACT_REPORTS.get('B1',{}).get('contract_status','NOT_EVALUABLE')}`.",
 f"- B3 model contract: `{MODEL_CONTRACT_REPORTS.get('B3',{}).get('contract_status','NOT_EVALUABLE')}`.",
 f"- Fault-onset status: `{ONSET_REPORT.get('status')}`.",
 f"- Notebook 08 family/scores join: `{FAMILY_AUDIT.get('status')}`.",
 f"- Frozen threshold provenance: `{THRESHOLD_REPORT.get('source_status')}`; value used for descriptive metrics: `{THRESHOLD_REPORT.get('threshold')}`.",
 '- No PINN inference or training was performed.',
 '- Static signal detector metrics, when produced, are not early-warning lead-time metrics.',
 '', '## Gate summary', ''
]
for k,v in GATES.items(): lines.append(f'- `{k}`: **{v}**')
lines += ['', '## Critical missing evidence', '']
lines += [f'- {x}' for x in missing_critical] if missing_critical else ['- No critical missing evidence detected by configured checks; review all source and semantic limitations before proceeding.']
lines += ['', '## Required next work', ''] + [f'{i+1}. {x}' for i,x in enumerate(final_report['next_actions'])]
lines += ['', '## Scope and limitations', '',
 'This notebook audits local/Drive artifacts and does not modify frozen Notebooks 04–08 or source data. Header/sample checks are bounded and do not prove the full multi-gigabyte CSV is globally clean. Safe checkpoint loading is metadata inspection only. Simulation-only results do not establish real-world EV battery safety.']
(RUN_DIR/'10_final_audit_report.md').write_text('\n'.join(lines),encoding='utf-8')

# Write artifact manifest only for files produced by this notebook; do not include itself to avoid recursive hash changes.
produced=[p for p in RUN_DIR.iterdir() if p.is_file() and p.name not in {'10_output_manifest.json'}]
manifest=[]
for p in sorted(produced):
    try: manifest.append({'path':str(p),'name':p.name,'size_bytes':p.stat().st_size,'sha256':sha256_file(p) if p.stat().st_size<=250*1024*1024 else None})
    except Exception as e: manifest.append({'path':str(p),'name':p.name,'error':f'{type(e).__name__}: {e}'})
(RUN_DIR/'10_output_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print((RUN_DIR/'10_final_audit_report.md').read_text(encoding='utf-8'))
print('\nOUTPUTS:')
for p in sorted(RUN_DIR.iterdir()): print(p.name, f'({p.stat().st_size:,} bytes)')

# Notebook 10 — Final Model Contract & Fault-Onset Audit

- **Overall readiness:** `BLOCKED_MISSING_CRITICAL_EVIDENCE`
- **Run directory:** `/content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1_model_contract_audit/run_20261015_075459`

## Key conclusions
- Notebook 06 source contract status: `NOT_EVALUABLE_SOURCE_MISSING`; input/output dimensions: `None -> None`.
- B1 model contract: `NOT_EVALUABLE_MODEL_CONTRACT`.
- B3 model contract: `NOT_EVALUABLE_MODEL_CONTRACT`.
- Fault-onset status: `NOT_EVALUABLE_MISSING_FAULT_ONSET_TIMESTAMP`.
- Notebook 08 family/scores join: `VALIDATED_JOIN_AND_SPLIT_AUDIT`.
- Frozen threshold provenance: `READ_FROM_NOTEBOOK08_SAVED_SUMMARY`; value used for descriptive metrics: `4.399034460052955`.
- No PINN inference or training was performed.
- Static signal detector metrics, when produced, are not early-warning lead-time metrics.

## Gate summary

- `project_artifacts_found`: **FAIL**
- `notebook06_source_contract_extracted`: **FAIL**
- `b1_inference

## Interpretation and stop condition

If the model contract is unresolved, do not add inference cells to this notebook. If onset ground truth is missing, do not compute lead-time metrics. The next scientific experiment is a separate, versioned temporal fault-injection dataset whose generator records the exact injection event and provides healthy pre-onset history; it must not rewrite `data/v1`.